# Final Project Notebook (Outline)

> **Purpose:** This notebook is the clean, story-driven final submission outline, aligned to `submissions/final_requirements.txt` and grounded in the pipeline outputs and experiment artifacts.

**Author:** 831004628  
**Course Project:** ATP Match Outcome Modeling


## 0. Executive Summary (to complete)

- **Motivation (1 paragraph):** Why ATP match prediction matters.
- **Research question (1 sentence):** Clear and measurable prediction/analysis objective.
- **Top findings (3 bullets):** Most important outcomes from modeling + experiments.
- **Practical takeaway (1 bullet):** What a coach/analyst could do with these results.


## 1. Motivation and Research Question

### 1.1 Motivation
ATP match outcomes are highly consequential for coaching decisions, betting markets, and tournament strategy, but many pre-match judgments are still made heuristically. This project turns that judgment process into a reproducible prediction workflow by using only information that would be available **before** the match starts (rank/race gaps, surface/court context, and temporal Elo features).

A central motivation is methodological: tennis data is naturally time-ordered, so leakage can quietly inflate performance if future matches influence past features. The pipeline is therefore designed around chronological safety (feature generation from past-only history) so that reported metrics reflect realistic forecasting quality rather than retrospective overfitting.

### 1.2 Final research question
**How accurately can we predict whether Team1 wins an ATP singles match using leakage-safe pre-match features (ranking/race differentials, context variables, and temporal Elo signals), and does adding temporal feature enrichment materially improve discrimination over a data-only baseline?**

### 1.3 Success criteria
- **Primary predictive quality:** achieve test ROC-AUC around or above **0.70** while maintaining test accuracy around the observed **~0.64-0.65** range from pipeline experiments.
- **Stability/generalization:** keep train/validation/test gaps modest (no major overfitting spikes), especially when comparing decision tree, random forest, and GBDT runs.
- **Experimental value-add:** show that temporal enrichment (Elo + clustering context) is at least comparable to, and ideally better than, the core data-only feature set on ROC-AUC.
- **Interpretability:** preserve a clear directional story that larger `rank_diff`/`elo_diff_team1` values correspond to higher Team1 win probability in aggregate analyses.



## 2. Data Overview and Scope

This project predicts **binary match outcomes** using a leakage-safe, pre-match feature table produced by the pipeline. The data scope and modeling grain are intentionally narrow so that reported metrics remain interpretable and reproducible.

### 2.1 Data source and temporal span
- Raw source: yearly ATP match CSV files in `data/csv_data/` (`atp_2000.csv` through `atp_2026.csv`).
- Modeling source: curated pipeline output in `data/processed/model_table.parquet`.
- Observed modeling span in the processed table: matches dated **2000-01-03 to 2026-02-09**.

### 2.2 Unit of analysis
- One row corresponds to **one singles match** with a Team1/Team2 representation.
- The table currently contains **92,112 rows** and **54 columns**.
- Team-side features are aligned to preserve pre-match semantics (e.g., rank and Elo are measured before match outcome is known).

### 2.3 Target variable and prediction task
- Target: `team1_wins` (1 if Team1 wins, else 0).
- Class balance is close to even (`team1_wins` mean ≈ **0.502**), which supports direct model comparison without heavy class-reweighting assumptions.
- Core predictive signal in this project comes from rank and temporal Elo features (`rank_diff`, `elo_diff_team1`, `elo_prob_team1_pre`) plus match context.

### 2.4 Feature scope used in experiments
- **Static engineered differentials:** rank/race, age, height, points, handedness, and related pairwise differences.
- **Temporal features:** pre-match Elo ratings/probability generated chronologically.
- **Context fields:** `surface_context` and `court_context`.
- The final table confirms strong coverage for key model features used in experiments (`rank_diff` and `elo_diff_team1` are complete in the processed table).

### 2.5 Data quality notes and limitations
- `surface_context` has a small `Unknown` category, but most matches are on Hard/Clay/Grass.
- `court_context` is effectively unavailable in the current processed data (`Unknown` for all rows), limiting fine-grained court-type interpretation.
- As with any long-horizon sports dataset, rule, equipment, and competitive-era changes may introduce temporal drift; this motivates time-aware evaluation in the experiments section.



In [6]:
from pathlib import Path
import pandas as pd
import os
print(os.getcwd())

model_table_path = Path('../data/processed/model_table.parquet')
df = pd.read_parquet(model_table_path)

print('Rows, columns:', df.shape)
print('Target distribution (team1_wins):')
print(df['team1_wins'].value_counts(normalize=True).rename('proportion'))

df[['match_date', 'team1_wins', 'rank_diff', 'elo_diff_team1', 'surface_context']].head()


/Users/keegansmith/Coding/school/Data-Mining-Project/submissions
Rows, columns: (92112, 54)
Target distribution (team1_wins):
team1_wins
1    0.502291
0    0.497709
Name: proportion, dtype: Float64


,match_date,team1_wins,rank_diff,elo_diff_team1,surface_context
0,2000-01-03,0,-28.0,0.702399,Hard
1,2000-01-03,0,-15.0,0.033908,Hard
2,2000-01-03,1,-38.0,0.000000,Hard
3,2000-01-03,0,47.0,0.736307,Hard
4,2000-01-03,0,46.0,0.000000,Hard


## 3. Pipeline Walkthrough (mapped to implementation)

This section explains how the modeling table is constructed from raw match records in a **chronologically safe** way, and why each stage is needed for the experiments in Section 4.

### 3.1 Raw ingestion, schema cleaning, and value normalization
The pipeline begins by loading yearly ATP source files and standardizing schema so downstream feature code can rely on stable field names and dtypes. Early cleaning stages remove malformed or unusable rows, reconcile column conventions across seasons, and enforce typed date/numeric columns.

Why this matters for experiments:
- The model comparison is only meaningful if all models train from the **same cleaned population**.
- Consistent dtypes prevent silent train/test skew (for example, numeric columns accidentally treated as strings in one split).

Implementation mapping:
- `src/tennis_pipeline/steps/01_load_raw.py`
- `src/tennis_pipeline/steps/02_clean_schema.py`
- `src/tennis_pipeline/steps/03_clean_values.py`

### 3.2 Canonical role assignment and supervised target construction
Tennis match records often contain player-role asymmetries (player A/B ordering). The role-splitting stage builds a canonical team1/team2 view and creates the binary target `team1_wins` used by all classifiers.

Why this matters for experiments:
- Without canonical roles, feature signs become inconsistent (for example, `rank_diff` can invert semantics).
- A single target definition ensures fair comparison between baseline and enhanced feature sets.

Implementation mapping:
- `src/tennis_pipeline/steps/04_split_roles.py`

### 3.3 Static pre-match feature engineering
Static features derive pre-match information from rank/race/context fields and pairwise differences. This includes core differentials such as `rank_diff` and `abs_rank_diff`, plus categorical context like `surface_context` and `court_context`.

Why this matters for experiments:
- These columns form the baseline signal in the “data-only” feature set.
- They also remain foundational in the enhanced feature set (enhanced = baseline + temporal and clustering enrichments).

Implementation mapping:
- `src/tennis_pipeline/steps/05_build_features_static.py`

### 3.4 Temporal feature engineering (leakage-safe)
Temporal enrichment computes pre-match strength estimates using chronological history only. The Elo stage adds features such as `elo_diff_team1` and `elo_prob_team1_pre`, while rolling windows summarize recent form.

Critical leakage control:
- Features are computed in match-date order and only from prior matches.
- The goal is to estimate what would have been known **at prediction time**, not after the fact.

Why this matters for experiments:
- Section 4’s uplift analysis (baseline vs enhanced) depends on this being leakage-safe; otherwise any gain would be unreliable.

Implementation mapping:
- `src/tennis_pipeline/steps/06_build_features_temporal_elo.py`
- `src/tennis_pipeline/steps/06b_build_features_temporal_rolling.py`

### 3.5 Optional unsupervised augmentation (clustering)
An additional branch builds clustering-oriented representations for player/match context and evaluates KMeans settings (tracked in the tuning artifact).

Why this matters for experiments:
- This stage supports the clustering hypothesis tested in Section 4.
- It allows us to assess whether unsupervised structure adds predictive value beyond rank + temporal signals.

Implementation mapping:
- `src/tennis_pipeline/steps/06c_build_features_clustering.py`
- `data/processed/clustering_tuning_artifact.json`

### 3.6 Final model-table assembly and feature contract
The finalization step selects leakage-safe model columns, preserves chronology metadata, and writes the training table consumed by experiment runners. In this notebook, the loaded model table confirms presence of key features used in later analysis (`rank_diff`, `abs_rank_diff`, `elo_diff_team1`, `elo_prob_team1_pre`, `surface_context`, `court_context`, `team1_wins`).

Why this matters for experiments:
- Section 4 compares models on a common feature contract and split logic.
- Reproducibility depends on a deterministic, documented final table artifact.

Implementation mapping:
- `src/tennis_pipeline/steps/07_finalize_model_table.py`
- `src/tennis_pipeline/experiments/feature_sets.py`
- `src/tennis_pipeline/experiments/model_training.py`
- `docs/pipeline_mapping.md`

---
**Bridge to Section 4:** The experiment story is therefore: hold cleaning/target construction constant, vary feature-set richness (data-only vs temporal+clustering), and evaluate whether added complexity improves generalization and calibration under the same pipeline contract.



In [7]:
# Optional: quick feature group inspection
feature_groups = {
    'core_rank_elo': ['rank_diff', 'abs_rank_diff', 'elo_diff_team1', 'elo_prob_team1_pre'],
    'context': ['surface_context', 'court_context'],
    'target': ['team1_wins'],
}
for group, cols in feature_groups.items():
    present = [c for c in cols if c in df.columns]
    print(f"{group}: {present}")


core_rank_elo: ['rank_diff', 'abs_rank_diff', 'elo_diff_team1', 'elo_prob_team1_pre']
context: ['surface_context', 'court_context']
target: ['team1_wins']


## 4. Experiment Design and Results Story

This section compares a baseline feature set against an enhanced feature set, then checks whether unsupervised clustering adds signal that is useful for downstream prediction.

### 4.1 Baseline vs enhanced models

I ran the same three model families (Decision Tree, Gradient Boosting, Random Forest) on two feature sets:

- **Baseline (`data_only`)**: rank/context pre-match fields.
- **Enhanced (`data_plus_temporal_elo_clustering`)**: baseline + temporal Elo differentials + cluster-based context from Elo dynamics.

Across all three model families, the enhanced feature set improved both discrimination and calibration on the held-out test split. The strongest result came from **Random Forest + enhanced features**:

- Test Log Loss improved from **0.6376 → 0.6238**.
- ROC-AUC improved from **0.6896 → 0.7070**.
- Brier Score improved from **0.2235 → 0.2175**.
- ECE (10 bins) improved from **0.0344 → 0.0201**.
- Accuracy improved from **0.6329 → 0.6481**.

These gains are consistent with the project hypothesis that temporal strength signals (Elo trend/differentials) capture form and matchup quality beyond static rank alone.

### 4.2 Clustering experiment (from processed artifacts)

The clustering experiment used **KMeans** on Elo-derived columns (`elo_diff_pre`, `elo_diff_team1`, `elo_prob_team1_pre`, `elo_team1_pre`, `elo_team2_pre`) with **train-only fitting** to prevent leakage.

- Best selected configuration: **k = 6**.
- Best silhouette score: **0.3648** (coarse stage).

Interpretation: clusters provide a compact summary of matchup archetypes (e.g., balanced vs strongly favored contests). Even if silhouette is moderate (as expected in noisy sports data), adding this representation inside the enhanced set aligns with the consistent improvement in downstream predictive metrics.

### 4.3 Model comparison table and confidence in results

The table below highlights the top two models on the enhanced set by log loss:

1. **Random Forest (best overall)**
2. **GBDT (runner-up)**

Confidence considerations:

- Improvements are **directionally consistent across multiple metrics**, not just one optimized objective.
- Validation and test accuracies track closely, which lowers concern about severe overfitting.
- Hyperparameter search found similar validation log-loss minima for RF and GBDT (~0.616), suggesting the ranking is stable but close; I therefore treat RF as the primary model and GBDT as a credible backup.



In [9]:
import json
from pathlib import Path

import pandas as pd

# --- Clustering artifact summary ---
artifact_path = Path('../data/processed/clustering_tuning_artifact.json')
artifact = json.loads(artifact_path.read_text())

print('Clustering method:', artifact.get('method'))
print('Fit scope:', artifact.get('fit_scope'))
print('Selected columns:', artifact.get('selected_source_columns'))
print('Chosen kmeans config:', artifact.get('kmeans'))

kmeans_results = pd.DataFrame(artifact.get('kmeans_results', []))
print('Top silhouette configurations:')
display(kmeans_results.sort_values('silhouette_score', ascending=False).head(5))

# --- Baseline vs enhanced comparison ---
comparison = pd.read_csv('../data/processed/model_training_feature_sets/feature_set_probability_metric_comparison.csv')
key_cols = [
    'feature_set', 'model', 'test_log_loss', 'test_roc_auc',
    'test_brier_score', 'test_ece_10_bins', 'test_accuracy'
]
print('Baseline vs enhanced feature sets (all models):')
display(comparison[key_cols].sort_values(['model', 'feature_set']))

# --- Best and runner-up on enhanced set ---
enhanced = comparison[comparison['feature_set'] == 'data_plus_temporal_elo_clustering'].copy()
leaders = enhanced.sort_values(['test_log_loss', 'test_roc_auc'], ascending=[True, False]).head(2)
print('Best and runner-up (enhanced set):')
display(leaders[key_cols])

# --- Hyperparameter tuning support ---
hp_best = pd.read_csv('../data/processed/model_training_hyperparameter_tuning/hyperparameter_tuning_best.csv')
print('Best hyperparameter configs by model:')
display(hp_best[['model', 'max_depth', 'min_samples_leaf', 'n_estimators', 'validation_accuracy', 'validation_log_loss']])



Clustering method: kmeans
Fit scope: train_only
Selected columns: ['elo_diff_pre', 'elo_diff_team1', 'elo_prob_team1_pre', 'elo_team1_pre', 'elo_team2_pre']
Chosen kmeans config: {'n_clusters': 6}
Top silhouette configurations:


,n_clusters,silhouette_score,stage
3,6.0,0.364810,coarse
0,2.0,0.361931,coarse
4,7.0,0.359152,local_refine
2,5.0,0.355104,local_refine
6,10.0,0.342748,coarse


Baseline vs enhanced feature sets (all models):


,feature_set,model,test_log_loss,test_roc_auc,test_brier_score,test_ece_10_bins,test_accuracy
0,data_only,decision_tree,0.683451,0.675152,0.230223,0.050146,0.627999
1,data_plus_temporal_elo_clustering,decision_tree,0.663498,0.694643,0.223608,0.037651,0.639887
2,data_only,gbdt,0.657142,0.676795,0.230771,0.061327,0.625556
3,data_plus_temporal_elo_clustering,gbdt,0.638870,0.695225,0.223434,0.049292,0.636630
4,data_only,random_forest,0.637562,0.689600,0.223537,0.034414,0.632885
5,data_plus_temporal_elo_clustering,random_forest,0.623808,0.707009,0.217525,0.020073,0.648138


Best and runner-up (enhanced set):


,feature_set,model,test_log_loss,test_roc_auc,test_brier_score,test_ece_10_bins,test_accuracy
5,data_plus_temporal_elo_clustering,random_forest,0.623808,0.707009,0.217525,0.020073,0.648138
3,data_plus_temporal_elo_clustering,gbdt,0.638870,0.695225,0.223434,0.049292,0.636630


Best hyperparameter configs by model:


,model,max_depth,min_samples_leaf,n_estimators,validation_accuracy,validation_log_loss
0,gbdt,3,5,100,0.652752,0.616114
1,random_forest,9,5,300,0.653946,0.616456


## 5. Error Analysis and Interpretation

### 5.1 Where the model succeeds
- Match contexts where predictions are most reliable.

### 5.2 Where the model struggles
- Upsets, sparse metadata, or cold-start players.

### 5.3 Feature interpretation
- Discuss directional effects of rank and Elo differences.
- Explain context effects (surface/court) if meaningful.


## 6. Conclusions

- Directly answer the research question.
- Summarize what evidence supports the answer.
- State practical implications and caveats.


## 7. Future Work

- Calibrated probabilities and decision thresholds.
- Tournament-level or player-form temporal windows.
- Better handling of missing context fields.


## 8. Reproducibility Checklist

- [ ] Confirm notebook runs top-to-bottom on clean environment.
- [ ] Keep only final narrative cells (remove dead ends).
- [ ] Ensure all claims in text are backed by displayed outputs.
- [ ] Verify consistency with `submissions/final_requirements.txt`.
